# PONITA + ECT, 5000 epochs (resumable)

Six models on MD17 aspirin, seed 0, 5000 epochs, target normalization on:

| Cell | Model | What it is |
|---|---|---|
| 5 | baseline | PONITA, no ECT |
| 6 | concat | global ECT token concatenated to every atom's input |
| 7 | film-final | global ECT token modulates the final molecule embedding (FiLM) |
| 8 | leap | each atom's own local ECT concatenated to its input |
| 9 | hybrid-adaln | AdaLN in every block driven by the **global** ECT + global ECT concat before the head |
| 10 | hybrid-leap | leap (each atom's local ECT concatenated to its input) + global ECT concat before the head. No AdaLN |

**What is saved per model** (in `CKPT_DIR` on Drive):
- `*_last.pt`: full training state every 25 epochs, so a disconnect resumes where it stopped.
- `*.pt`: best checkpoint on **validation forces**. `*_bestE.pt`: best checkpoint on **validation energy**.
- `*_history.json`: train E/F every epoch, val E/F every 25 epochs (for the curves in Cell 11).
- `*_results.json`: val and test MAE for both best checkpoints (for the table in Cell 12).

Checkpoints go to a **new folder** (`dino_ckpts_5000`), so your 2000-epoch results are not touched.

**Compute:** each run is 2.5× as long as a 2000-epoch run. Six of them is about 15 times one of your earlier runs.
Run the cells one at a time, not "Run all".

**If Colab disconnects:** reconnect, run Cells 1–3 again, then the cell that was running. It prints
`Resumed from ..._last.pt at epoch N` and continues.

### One-time setup
The repo on Drive (`MyDrive/dino`) must have the latest `train_ponita_md17.py`, `models/ponita_ect.py`,
`models/egnn_ect.py` and `qm9_ect.py`.

In [ ]:
# Cell 1 — confirm you have a GPU
!nvidia-smi

In [ ]:
# Cell 2 — mount Drive, set paths and shared settings (re-run after any reconnect)
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/dino'
CKPT_DIR = '/content/drive/MyDrive/dino_ckpts_5000'   # new folder: 2000-epoch runs stay untouched
DATA_DIR = '/content/drive/MyDrive/dino_data/md17'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
assert os.path.exists(os.path.join(REPO_DIR, 'train_ponita_md17.py')), \
    f'train_ponita_md17.py not found in {REPO_DIR} — fix REPO_DIR'

EPOCHS = 5000
SEED   = 0
COMMON = (f'--molecule aspirin --epochs {EPOCHS} --seed {SEED} '
          f'--data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" '
          '--log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 '
          '--lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 '
          '--basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 --icosphere-nu 3')
os.chdir(REPO_DIR)
print('Repo :', REPO_DIR)
print('Ckpts:', CKPT_DIR)
print('Data :', DATA_DIR)

In [ ]:
# Cell 3 — install dependencies matching Colab's torch build
# (ponita needs Python 3.10+, which Colab has — so NONE of the cluster's
#  match/case patches are needed here.)
import torch
v = torch.__version__.split('+')[0]
cu = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
wheel = f'https://data.pyg.org/whl/torch-{v}+{cu}.html'
print('torch', torch.__version__, '->', wheel)
!pip -q install pyg-lib -f {wheel}
!pip -q install torch-scatter -f {wheel}
!pip -q install torch-geometric
!pip -q install git+https://github.com/ebekkers/ponita.git

In [ ]:
# Cell 4 — sanity check: imports work and the ECT is rotation-invariant
os.chdir(REPO_DIR)
!python test_equivariance.py

In [ ]:
# Cell 5 — baseline (resumes automatically). PONITA without ECT.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/baseline_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py --model baseline {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 6 — concat (resumes automatically). Global ECT token concatenated at the input.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/concat_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py --model ect --cond-type concat {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 7 — film-final (resumes automatically). Global ECT token, FiLM on the final embedding.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/film-final_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py --model ect --cond-type film-final {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 8 — leap (resumes automatically). Per-atom local ECT concatenated at the input (5 Å).
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/leap_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py --model ect --cond-type leap {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 9 — hybrid-adaln (resumes automatically). AdaLN in every block from the global ECT, plus global ECT concat before the head.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/hybrid-adaln_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py --model ect --cond-type hybrid-adaln {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 10 — hybrid-leap (resumes automatically). leap (per-atom local ECT at the input), plus global ECT concat before the head. No AdaLN.
os.chdir(REPO_DIR)
LOG = f'{CKPT_DIR}/hybrid-leap_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py --model ect --cond-type hybrid-leap {COMMON} 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 11 — training curves for every model that has started (re-run any time)
import glob, json, re
import numpy as np
import matplotlib.pyplot as plt

# Fixed colour per model, so a model keeps its colour whichever runs exist.
STYLE = {
    'baseline':     dict(color='#8a8880', ls='--'),
    'concat':       dict(color='#2a78d6', ls='-'),
    'film-final':   dict(color='#eb6834', ls='-'),
    'leap':         dict(color='#1baf7a', ls='-'),
    'hybrid-adaln': dict(color='#eda100', ls='-'),
    'hybrid-leap':  dict(color='#e87ba4', ls='-'),
}

def model_name(path):
    base = os.path.basename(path)
    if 'PONITA_baseline' in base:
        return 'baseline'
    m = re.search(r'ECT_(.+?)_aspirin', base)
    return m.group(1) if m else base

def smooth(y, k=25):
    # Train MAE is logged every epoch and is noisy; show a 25-epoch moving average.
    y = np.asarray(y, dtype=float)
    if len(y) < k:
        return y
    return np.convolve(y, np.ones(k) / k, mode='valid')

hists = {}
for path in sorted(glob.glob(f'{CKPT_DIR}/*_history.json')):
    name = model_name(path)
    if name in STYLE:
        with open(path) as f:
            hists[name] = json.load(f)
assert hists, f'No *_history.json in {CKPT_DIR} yet — start a training cell first.'
order = [m for m in STYLE if m in hists]

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), sharex=True)
panels = [
    (axes[0, 0], 'train_f', 'Train force MAE (meV/Å), 25-epoch average'),
    (axes[0, 1], 'val_f',   'Validation force MAE (meV/Å)'),
    (axes[1, 0], 'train_e', 'Train energy MAE (meV), 25-epoch average'),
    (axes[1, 1], 'val_e',   'Validation energy MAE (meV)'),
]
for ax, key, title in panels:
    for name in order:
        h = hists[name]
        if key.startswith('train'):
            y = smooth(h[key])
            x = np.asarray(h['epoch'][len(h['epoch']) - len(y):])
        else:
            x, y = np.asarray(h['val_epoch']), np.asarray(h[key])
        if len(y):
            ax.plot(x, y, lw=1.8, label=name, **STYLE[name])
    ax.set_title(title, fontsize=11, loc='left')
    ax.set_yscale('log')
    ax.grid(True, which='both', color='#e4e2dc', lw=0.6)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
for ax in axes[1]:
    ax.set_xlabel('epoch')
axes[0, 1].legend(frameon=False, fontsize=9, loc='upper right')
fig.suptitle(f'PONITA + ECT on MD17 aspirin, seed {SEED} ({EPOCHS} epochs)', x=0.01, ha='left', fontsize=13)
fig.tight_layout()
out = f'{CKPT_DIR}/training_curves_seed{SEED}.png'
fig.savefig(out, dpi=150)
plt.show()
print('Saved', out)
print('Epochs done:', {m: len(hists[m]['epoch']) for m in order})

In [ ]:
# Cell 12 — results table: test MAE at the best-force and best-energy checkpoints
import glob, json, re
import pandas as pd

rows = []
for path in sorted(glob.glob(f'{CKPT_DIR}/*_results.json')):
    with open(path) as f:
        r = json.load(f)
    name = 'baseline' if 'baseline' in r['label'] else re.search(r'\((.+)\)', r['label']).group(1)
    for which, ckpt in (('best_f', 'best val F'), ('best_e', 'best val E')):
        if which in r:
            rows.append({'model': name, 'checkpoint': ckpt,
                         'E (meV)': r[which]['test_e'], 'F (meV/Å)': r[which]['test_f']})
assert rows, 'No *_results.json yet — they are written when a model finishes training.'

df = pd.DataFrame(rows)
base = df[df.model == 'baseline'].set_index('checkpoint')
if len(base):
    df['Δ E'] = df.apply(lambda r: r['E (meV)'] / base.loc[r.checkpoint, 'E (meV)'] - 1, axis=1)
    df['Δ F'] = df.apply(lambda r: r['F (meV/Å)'] / base.loc[r.checkpoint, 'F (meV/Å)'] - 1, axis=1)
order = ['baseline', 'concat', 'film-final', 'leap', 'hybrid-adaln', 'hybrid-leap']
df['model'] = pd.Categorical(df.model, [m for m in order if m in set(df.model)] +
                             sorted(set(df.model) - set(order)))
df = df.sort_values(['checkpoint', 'model'], ascending=[False, True]).reset_index(drop=True)
fmt = {'E (meV)': '{:.2f}', 'F (meV/Å)': '{:.2f}', 'Δ E': '{:+.1%}', 'Δ F': '{:+.1%}'}
display(df.style.format({k: v for k, v in fmt.items() if k in df}).hide(axis='index'))
df.to_csv(f'{CKPT_DIR}/results_seed{SEED}.csv', index=False)